In [ ]:
%pip install "sagemaker>=2.0,<3.0" -q
!pip install "sagemaker<3" pyathena awswrangler  --quiet
!pip install 'boto3>1.17.21' -q

# Set up Athena Pneumonia DB
This notebook is purely for a user to register the pneumonia db in thier Glue Catalog

In [ ]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

sess = sagemaker.Session()
default_bucket = sess.default_bucket()
region = boto3.Session().region_name
bucket = "pneumonia-data-set-group-4"

# Athena staging directory (uses YOUR default bucket for query results)
s3_staging_dir = f"s3://{default_bucket}/athena/staging"

# Connect to Athena
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

print(f"Region: {region}")
print(f"Data bucket: {bucket}")
print(f"Staging dir: {s3_staging_dir}")

In [ ]:
database_name = "pneumonia_db"

### Drop the DB if it exists

In [ ]:
drop_statement = f"""
DROP TABLE IF EXISTS {database_name}.image_metadata
"""
pd.read_sql(drop_statement, conn)

## Create table

In [ ]:
statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.image_metadata (
    image_id              STRING,
    raw_s3_key            STRING,
    preprocessed_s3_key   STRING,
    label                 STRING,
    label_int             TINYINT,
    split                 STRING,
    source                STRING,
    file_type             STRING,
    pixel_mean            DOUBLE,
    pixel_std             DOUBLE,
    img_height            INT,
    img_width             INT,
    event_time            STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{bucket}/pneumonia-project/metadata/'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
pd.read_sql(statement, conn)
print("✅ Table created!")